# Import package

In [ ]:
import os
import argparse
import scanpy as sc

from cellia import *
from cellia_web import *

# Load dataset

In [ ]:
adata = sc.read_h5ad("dataset/YOUR_DATASET.h5ad")
adata

# Configure CELLIA parameters

In [ ]:
gpt_api_key = "GPT_API_KEY"
gemini_api_key = "GEMINI_API_KEY"

gpt_model = "gpt-4.1-2025-04-14"
gemini_model = "gemini-2.5-flash"

In [ ]:
tissue_db = "PBMC|blood"
tissue_type = "human PBMC"

# Run CELLIA

## Step A: Identification of DEG

In [ ]:
find_markers(adata=adata)

adata

## Step B-C: Filtering DEG and selection of top-k markers

- Major cell type annotaiton

In [ ]:
filter_markers(
    adata=adata,
    mode="db",
    tissue_db=tissue_db,
    k=15
)

adata

- Subtype-level cell type annotation

In [ ]:
filter_markers(
    adata=adata,
    tissue_db=tissue_db,
    subset_db= "dendritic|DC",
    k=15,
    deg_mode="subset",
    mode="subset_db"
)

## Step D: LLM-based cell type annotaiton

- Major cell type annotation

In [ ]:
gpt_anno(
    adata=adata,
    api_key=gpt_api_key,
    tissue_type=tissue_type,
    mode="major",
    model=gpt_model
)

adata

- Subtype-level cell type annotation

In [ ]:
gpt_anno(
    adata=adata,
    api_key=gpt_api_key,
    tissue_type=tissue_type,
    model=gpt_model,
    mode="subset", 
    parent_celltype="Dendritic cells"
)

adata

## Step E: Interactive visualization

In [ ]:
launch_cap_style_app(
    adata=adata,
    port=8050,
    debug=True,
    rationale_json_path="cellia_output/gpt_explanations_subset.json",
    num_top_k=15,
    use_uns_markers="marker_list_subset",
    use_uns_llm_annotation="GPT_annotation_subset"
)